# Spaces Pipeline Pro — SFT + Multi-step GRPO (Unsloth)

Train an LLM agent on the [Spaces Pipeline Pro OpenEnv environment](https://github.com/rishabh16196/hf-space-composer).

**Hackathon theme fit**: Theme #2 (Long-Horizon Planning) + Theme #3.1 (Tool Discovery / World Modeling).

**What this notebook does**:
1. Installs the env + Unsloth + PEFT
2. Pulls the pre-trained SFT LoRA adapter from HF (`rishabh16196/spaces-pipeline-sft-1.5b`)
3. Runs multi-step GRPO with Unsloth (model generates at every env step)
4. Evaluates on the two-tier held-out (5 easy + 5 hard tasks)
5. Plots the reward curve

**Hardware**: requires a CUDA GPU (A100 / L4). Select **Runtime → Change runtime type → GPU** before running.

## 1. Install dependencies

Strategy: upgrade torch to 2.5.1 first (Unsloth requires ≥2.5), then install Unsloth with `--no-deps` so it doesn't try to pull a nightly torch build. This avoids the flash-attn source-rebuild trap.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!pip install -q --upgrade pip
# Pin torch to 2.5.1 (Unsloth-compatible)
!pip install -q --upgrade torch==2.5.1 torchvision==0.20.1 --index-url https://download.pytorch.org/whl/cu124
# Core training deps
!pip install -q 'peft>=0.13.0' 'datasets>=3.0.0' 'accelerate>=0.34.0' 'transformers>=4.46.0' 'safetensors>=0.4.0' 'bitsandbytes>=0.43.0' matplotlib
# Unsloth (no-deps) + xformers (ABI-matched, no-deps) to prevent torch downgrade
!pip install -q --no-deps --no-build-isolation 'xformers==0.0.28.post2' --index-url https://download.pytorch.org/whl/cu124
!pip install -q --no-deps --no-build-isolation unsloth unsloth_zoo cut_cross_entropy tyro
# Verify
import torch
print('torch:', torch.__version__, '| cuda:', torch.cuda.is_available())
from unsloth import FastLanguageModel
print('unsloth OK')

## 2. Clone the env repo + install

In [ ]:
!git clone --depth=1 https://github.com/rishabh16196/hf-space-composer.git
%cd hf-space-composer
!pip install -q -e .

## 3. Download the SFT warmstart adapter

We already trained SFT locally on Apple Silicon (Qwen 2.5 1.5B + LoRA, 474 pairs, 10 min). The resulting adapter is on HF Hub — Colab just pulls it.

In [ ]:
from huggingface_hub import snapshot_download
sft_path = snapshot_download(repo_id='rishabh16196/spaces-pipeline-sft-1.5b', local_dir='/tmp/sft_adapter')
print('SFT adapter →', sft_path)
!ls {sft_path}

## 4. Env smoke test — HeuristicAgent on a marathon task

The HeuristicAgent follows each task's `gold_pipeline` and hits the ~0.92 ceiling on marathons. This is our upper bound.

In [ ]:
from server.spaces_pipeline_environment import SpacesPipelineEnvironment
from inference import HeuristicAgent

env = SpacesPipelineEnvironment()
agent = HeuristicAgent()
obs = env.reset(seed=42, task='marathon_investigation_037')
agent.reset('marathon_investigation_037')
while not obs.done:
    action = agent.act(obs)
    if action is None:
        break
    obs = env.step(action)
print(f'grade={obs.grade_score:.3f}  steps={obs.step_number}  spaces_called={obs.spaces_called}')

## 5. Multi-step GRPO with Unsloth

**Why trajectory-level GRPO**: our env is multi-step. TRL's vanilla `GRPOTrainer` treats each prompt as a one-shot completion, which teaches the agent only the step-0 behavior. Empirically that formulation **regressed** our policy on held-out tasks. Instead, we use our own trajectory-level loop: the model generates an action at EVERY env step, the final trajectory grade becomes the group-relative advantage, and that advantage is assigned to every (prompt, action) pair in the trajectory.

Batched rollout: all B·G trajectories' next-step prompts are generated in a single `model.generate()` call per env iteration, using Unsloth's `for_inference` mode.

**Config** (A100, ~90 min for 100 steps):
- B=4 tasks/step × G=6 gens/task = 24 trajectories/step
- lr=3e-6, KL β=0.05 (protects SFT gains)
- update micro-batch=16

The script logs one line per step:
```
step 12/100  avg_r=0.68 [min=0.15 max=1.00]  n_traj=24 invalid=2 grp_skip=1  loss=-0.012 kl=+0.001  roll=52s upd=18s tot=70s
```

In [ ]:
%cd local_training
!python -u grpo_unsloth.py \
    --model Qwen/Qwen2.5-1.5B-Instruct \
    --adapter /tmp/sft_adapter \
    --output-dir outputs/grpo_colab \
    --max-steps 100 \
    --num-gens 6 \
    --batch-size 4 \
    --update-micro-batch 16 \
    --save-every 25 \
    --lr 3e-6 \
    --beta 0.05

## 6. Plot the reward + KL curves

In [ ]:
import json
import matplotlib.pyplot as plt

metrics = json.loads(open('outputs/grpo_colab/train_metrics.json').read())
steps = [m['step'] for m in metrics if not m.get('skipped')]
rewards = [m['avg_reward'] for m in metrics if not m.get('skipped')]
kls = [m.get('kl', 0) for m in metrics if not m.get('skipped')]

fig, ax = plt.subplots(1, 2, figsize=(14, 4))
ax[0].plot(steps, rewards, '-o', markersize=3)
ax[0].set_title('Avg trajectory reward per GRPO step')
ax[0].set_xlabel('step'); ax[0].set_ylabel('avg reward'); ax[0].set_ylim(0, 1)
ax[0].grid(True, alpha=0.3)
ax[1].plot(steps, kls, '-o', markersize=3, color='orange')
ax[1].set_title('KL divergence from SFT reference')
ax[1].set_xlabel('step'); ax[1].set_ylabel('KL')
ax[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Two-tier held-out evaluation

Compare Base Qwen vs SFT-only vs SFT+GRPO across 5 easy + 5 hard held-out tasks (none seen during training).

In [ ]:
!python -u eval_two_tier.py \
    --model Qwen/Qwen2.5-1.5B-Instruct \
    --adapter outputs/grpo_colab \
    --skip-heuristic

## 8. (Optional) Upload trained adapter to HF Hub

Uncomment and paste a write-token HF login if you want to publish.

In [ ]:
# from huggingface_hub import upload_folder, login
# login()  # paste your HF write token when prompted
# upload_folder(repo_id='YOUR_USER/spaces-pipeline-grpo-1.5b',
#               folder_path='outputs/grpo_colab',
#               commit_message='GRPO adapter from Colab')